# Lab 5: Discrete-Event Simulation of a Coffee Shop Queue

Complete the code TODOs, the required written responses, the manager recommendation, the reflection, and the AI-use statement. Think about the discussion questions and be prepared to explain your reasoning to the TA. Every code location that you must complete is marked `TODO`; code marked **Provided** or **do not edit** should be left unchanged. This notebook is both the practical work and the report.

## Student Information

**Name:** Write your answer here

**Student ID:** Write your answer here

**Date:** Write your answer here

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Provided lab settings.
SEED = 211          
N_CUSTOMERS = 500   # customers in every simulation run

## Part A: Event Logic

Chapter 7 defines a discrete-event simulation as a stochastic model whose state changes only at event times. For this one-server, first-come-first-served (FCFS) queue there are two events: an **arrival** (a customer enters the system) and a **departure** (a customer finishes service and leaves).

> **How this lab's code represents events.** The Chapter 7 pseudocode keeps an event list and repeatedly jumps the clock to the next arrival or departure. With one FCFS server there is a shortcut: customers are served in arrival order, so customer $i$ starts service at
> $$\texttt{start}_i=\max(\texttt{arrival}_i,\ \texttt{end}_{i-1}),\qquad \texttt{end}_i=\texttt{start}_i+\texttt{service}_i .$$
> This recursion produces exactly the same arrival and departure times as the event-list algorithm. Part C then rebuilds the ordered event sequence from these times to measure the queue.

Answer briefly:

1. Which event can increase queue length?

   **Response 1:** Write your answer here.

2. Which event frees the barista?

   **Response 2:** Write your answer here.

3. Why does customer $i$ depend on customer $i-1$'s service end time?

   **Response 3:** Write your answer here.

## Part B: Build an M/M/1 Queue

An M/M/1 queue has exponential interarrival times (arrival rate $\lambda$), exponential service times (service rate $\mu$), and one FCFS server. The rates are **per hour**, but this lab reports times in **minutes**. NumPy's `rng.exponential` takes the **mean** (`scale`), not the rate, so convert first:

$$
\text{mean interarrival}=\frac{60}{\lambda}\ \text{min},\qquad
\text{mean service}=\frac{60}{\mu}\ \text{min},\qquad
\rho=\frac{\lambda}{\mu}.
$$

The steady-state M/M/1 model exists only when $\rho<1$.

In [ ]:
def simulate_mm1(arrival_rate, service_rate, n_customers, seed=None):
    if arrival_rate <= 0 or service_rate <= 0 or n_customers <= 0:
        raise ValueError("Rates and n_customers must be positive")

    rng = np.random.default_rng(seed)

    # Generate all interarrival gaps first, then all service times (minutes).
    # Then convert the gaps to arrival times.
    interarrival = TODO
    service = TODO
    arrival = TODO

    start = np.empty(n_customers)
    end = np.empty(n_customers)

    for i in range(n_customers):
        previous_end = end[i - 1] if i > 0 else 0
        start[i] = TODO
        end[i] = TODO

    # Provided output-table code - do not edit.
    return pd.DataFrame({
        "customer": np.arange(1, n_customers + 1),
        "arrival": arrival,
        "service": service,
        "start": start,
        "end": end,
        "wait": start - arrival,
        "time_in_system": end - arrival,
    })

In [ ]:
# Run the Normal scenario (arrival rate 6, service rate 8 per hour)
normal_df = TODO

# Provided checks -- do not edit.
assert list(normal_df.columns) == [
    "customer", "arrival", "service", "start", "end", "wait", "time_in_system"
]
assert len(normal_df) == N_CUSTOMERS
normal_df.head()

### Trial checks

Before running the next cell, predict what each case should look like.

| Case | Arrival rate | Service rate | $\rho$ | Prediction |
|---|---:|---:|---:|---|
| Low load | 2 | 10 | 0.2 | most customers wait close to 0 |
| Near capacity | 7.8 | 8 | 0.975 | waiting is large and very variable from run to run |
| Overloaded | 10 | 8 | 1.25 | the queue tends to grow because work arrives faster than it is completed |

If your overloaded case looks calm, you have probably passed a **rate** where NumPy expects a **mean**. That gives mean gaps of 10 minutes and mean services of 8 minutes, which turns $\rho=1.25$ into $\rho=0.8$. This is the same mean-versus-rate trap as Lab 3 Part B.

In [ ]:
# Run the three diagnostic cases with N_CUSTOMERS customers and seed SEED.
trial_cases = {
    "Low load": (2.0, 10.0),
    "Near capacity": (7.8, 8.0),
    "Overloaded": (10.0, 8.0),
}

trial_rows = []
for name, (arrival_rate, service_rate) in trial_cases.items():
    result = TODO
    trial_rows.append({
        "case": name,
        "rho": arrival_rate / service_rate,
        "average_wait": result["wait"].mean(),
        "longest_wait": result["wait"].max(),
    })

# Provided table -- do not edit.
pd.DataFrame(trial_rows).set_index("case").round(3)

In [ ]:
# Repeat the near-capacity case (7.8, 8.0) with five different seeds.
near_rows = []
for seed in [1, 2, 3, 4, 5]:
    result = TODO
    near_rows.append({
        "seed": seed,
        "average_wait": result["wait"].mean(),
        "longest_wait": result["wait"].max(),
    })

# Provided table -- do not edit.
pd.DataFrame(near_rows).set_index("seed").round(2)

### Part B Questions

Think about the following questions:

1. Why does the overloaded case have no steady-state queue?
2. The five near-capacity runs use the same model. Why are their average waits so different?
3. Why must `rng.exponential` receive `60 / rate` rather than `rate`?

## Checkpoint 1: Event Logic and Queue Model (1 of 3 Points)


Show your notebook to the TA after completing Parts A-B. To receive the first 1 point, your notebook must contain:

1. the three Part A responses;
2. a working `simulate_mm1` and the Normal-scenario table with the required columns; and
3. the trial-check table and the five-seed near-capacity table.

Be prepared to explain the rate-to-mean conversion, the recursion `start = max(arrival, previous_end)`, and why $\rho\ge 1$ means there is no steady-state queue. Continue with Parts C-E after the checkpoint.

## Part C: Queue Output Measures

This lab defines **queue length** as the number of waiting customers, excluding the customer in service. When a departure and an arrival happen at the same time, the departure is processed first.

**Utilisation** is estimated as the fraction of time the barista is busy between the first arrival and the last departure (the variable `horizon`).

In [ ]:
def event_history(df):
    # Provided event reconstruction -- do not edit.
    events = pd.concat([
        pd.DataFrame({"time": df["arrival"], "delta": 1, "order": 1}),
        pd.DataFrame({"time": df["end"], "delta": -1, "order": 0}),
    ]).sort_values(["time", "order"], ignore_index=True)
    events["number_in_system"] = events["delta"].cumsum()
    events["queue_length"] = (events["number_in_system"] - 1).clip(lower=0)
    return events


def summarise_queue(df):
    events = event_history(df)
    horizon = df["end"].iloc[-1] - df["arrival"].iloc[0]

    # Calculate the five required outputs.
    return {
        "average_wait": TODO,
        "longest_wait": TODO,
        "average_time_in_system": TODO,
        "utilisation": TODO,
        "maximum_queue_length": TODO,
    }


normal_summary = pd.Series(summarise_queue(normal_df), name="Normal")
normal_summary

In [ ]:
# Provided queue-length plot -- do not edit.
normal_events = event_history(normal_df)
plt.figure(figsize=(9, 4))
plt.step(normal_events["time"], normal_events["queue_length"], where="post")
plt.xlabel("Time (minutes)")
plt.ylabel("Customers waiting")
plt.title("Normal-scenario queue length")
plt.show()

### Part C Questions

Think about the following questions:

1. The plot shows the queue repeatedly emptying and then building up in bursts. Why, when $\rho=0.75<1$?
2. Utilisation depends on the time window used. What would change if the window ended at the last arrival instead of the last departure?

## Part D: Scenario Comparison

Compare four demand levels with the same service rate. For a **stable** M/M/1 queue, Chapter 7 gives the steady-state mean waiting time in the queue:

$$
W_q=\frac{\rho}{\mu-\lambda}\ \text{hours}\qquad(\rho<1),
$$

so in minutes it is $60\,\rho/(\mu-\lambda)$. There is no steady-state value when $\rho\ge 1$.

First complete the theory function, then run every scenario with `N_CUSTOMERS` and `SEED`. Reusing the same seed in every scenario means each scenario is driven by the same underlying random numbers (common random numbers), so differences between rows come from the rates, not from luck.

In [ ]:
def mm1_theory_wait(arrival_rate, service_rate):
    # Steady-state mean wait in queue, in MINUTES; NaN when the queue is unstable.
    if arrival_rate >= service_rate:
        return np.nan
    rho = TODO
    return TODO


# Provided check -- do not edit.
assert np.isclose(mm1_theory_wait(6.0, 8.0), 22.5), "Check the hour-to-minute conversion."
assert np.isnan(mm1_theory_wait(9.0, 8.0))
print("Theory function check passed.")

In [ ]:
scenarios = {
    "Calm": {"arrival_rate": 4.0, "service_rate": 8.0},
    "Normal": {"arrival_rate": 6.0, "service_rate": 8.0},
    "Rush": {"arrival_rate": 7.5, "service_rate": 8.0},
    "Overloaded": {"arrival_rate": 9.0, "service_rate": 8.0},
}

scenario_rows = []
for name, settings in scenarios.items():
    result = TODO
    # Provided row-building code -- do not edit.
    scenario_rows.append({
        "scenario": name,
        "rho": settings["arrival_rate"] / settings["service_rate"],
        **summarise_queue(result),
        "theory_wait": mm1_theory_wait(settings["arrival_rate"], settings["service_rate"]),
        "run_hours": result["end"].iloc[-1] / 60,
    })

# Provided comparison-table code -- do not edit.
scenario_summary = pd.DataFrame(scenario_rows).set_index("scenario")
scenario_summary.round(3)

### Part D Questions

1. How do the average wait, longest wait, and maximum queue length change as $\rho$ approaches and then exceeds 1?

   **Response 1:** Write your answer here.

2. Compare `average_wait` with `theory_wait`. In which scenario is the gap largest? Give one reason why a single 500-customer run that starts with an empty shop can differ from the steady-state value.

   **Response 2:** Write your answer here.

3. Look at `run_hours`. Roughly how many hours of continuous operation does each run represent? What does this mean if we want to describe a real morning rush that lasts one or two hours?

   **Response 3:** Write your answer here.

Think about the following question:

4. The overloaded run finishes and produces finite numbers. Why does that not contradict the statement that the overloaded queue has no steady state?

## Part E: Manager Challenge

The manager wants to reduce waiting during rush periods and asks whether raising the service rate from 8 to 10 customers per hour (mean service time 7.5 to 6 minutes) is worth it.

One run is one possible history and can be lucky or unlucky (you saw this with the five near-capacity seeds). Chapter 7 notes that one run is useful for debugging, not for a final decision. So compare the two cases over **20 replications** (seeds 211 to 230), using the **same seeds for both cases**. This makes the comparison paired: each seed gives one rush run and one improved run driven by the same random numbers.

The provided summary uses the Lab 4 interval: mean $\pm\,1.96\times$ sample SD $/\sqrt{R}$.

In [ ]:
def replicate_average_wait(arrival_rate, service_rate, n_customers, seeds):
    # Return an array with one average wait per seed.
    waits = []
    for seed in seeds:
        result = TODO
        waits.append(TODO)
    return np.array(waits)


seeds = range(211, 231)   # Provided: 20 replications

rush_waits = TODO
improved_waits = TODO


# Provided summary code -- do not edit.
def replication_summary(waits, arrival_rate, service_rate):
    R = len(waits)
    mean = waits.mean()
    sd = waits.std(ddof=1)
    half_width = 1.96 * sd / np.sqrt(R)
    return {
        "replications": R,
        "mean_average_wait": mean,
        "sd": sd,
        "ci_low": mean - half_width,
        "ci_high": mean + half_width,
        "theory_wait": mm1_theory_wait(arrival_rate, service_rate),
    }


manager_comparison = pd.DataFrame([
    {"case": "Rush, service rate 8", **replication_summary(rush_waits, 7.5, 8.0)},
    {"case": "Improved, service rate 10", **replication_summary(improved_waits, 7.5, 10.0)},
]).set_index("case")

reduction = rush_waits - improved_waits
print(f"Mean reduction in average wait: {reduction.mean():.2f} minutes")
print(f"Seeds where the improvement reduced average wait: {(reduction > 0).sum()} of {len(reduction)}")
manager_comparison.round(2)

### Part E Questions

1. Does the 95% interval for the rush case contain `theory_wait`? If not, does that mean the simulator is wrong? Explain what adding replications fixes and what it does not fix.

   **Response 1:** Write your answer here.

2. By how much does the improvement reduce average waiting time, and how consistent is the reduction across seeds?

   **Response 2:** Write your answer here.

Think about the following question:

3. Why is it better to use the same seeds for both cases than different seeds?

## Recommendation, Reflection, and AI Use

**Manager recommendation:** Use the Part D and Part E evidence. State what operational change (for example, staff training, pre-preparing ingredients, a simpler rush menu, or a second preparation station) could realistically deliver the faster service rate, and state one important limitation of this model. Write your answer here.

**Reflection:** What makes this a discrete-event simulation? What happened as the arrival rate approached the service rate? Which output measure was most useful, and which modelling assumption is least realistic? Write your answer here.

**AI-use statement:** If AI was used, identify the tool, how it was used, and how you checked or corrected its output. If AI was not used, say so. You must be able to explain all submitted work. Write your answer here.

## Checkpoint 2: Final Submission

Show the completed Parts C-E, recommendation, reflection, and AI-use statement to the TA. Be prepared to answer one code question and one modelling question, for example why the rush simulation does not match $W_q$. After the final checkpoint:

1. save the notebook as `Lab05_StudentID.ipynb`, for example `Lab05_6900789.ipynb`;
2. restart the kernel and run all cells from top to bottom;
3. check that there are no errors;
4. save the notebook; and
5. upload the single completed `.ipynb` file to the **Lab 05** assignment in Google Classroom before leaving the lab.